In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv(r'C:\Users\ericl\OneDrive\Desktop\GitHub\Lending Club Loan Dataset 2007_2011\loan.csv')
# transform the issue_d column to datetime format and the int_rate column to float 
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%y")
df["int_rate"] = df["int_rate"].astype(str).str.rstrip("%").astype(float)

conn = sqlite3.connect('lending_club.db')
df.to_sql("loans",conn,if_exists='replace',index=False)

C:\Users\ericl\AppData\Local\Temp\ipykernel_31716\571195034.py:1: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\ericl\OneDrive\Desktop\GitHub\Lending Club Loan Dataset 2007_2011\loan.csv')


39717

In [4]:
query = """
SELECT loan_amnt, int_rate, grade 
FROM loans 
WHERE grade = 'A' AND loan_amnt > 20000 
ORDER BY loan_amnt DESC
"""

pd.read_sql(query, conn)

,loan_amnt,int_rate,grade
0,35000,8.90,A
1,35000,8.90,A
2,35000,8.90,A
3,35000,8.90,A
4,35000,8.90,A
...,...,...,...
292,20500,6.91,A
293,20500,6.54,A
294,20400,6.54,A
295,20400,7.74,A


In [5]:
# as expected, average interest rate increases as the grade decreases.
# most loans have grade A and B
query = """
SELECT grade, COUNT(*) as num_loans, AVG(int_rate) as avg_rate
FROM loans 
GROUP BY grade
ORDER BY grade
"""

pd.read_sql(query, conn)

,grade,num_loans,avg_rate
0,A,10085,7.335262
1,B,12020,11.021224
2,C,8098,13.552960
3,D,5307,15.719900
4,E,2842,17.711714
5,F,1049,19.749323
6,G,316,21.401044


In [6]:
#see, for each purpose, what is the number of loans and the charge off rate (i.e. the proportion of loans that were charged off).
#small businesses loans have the highest charge off rate, while loans for educational purposes have the lowest charge off rate.
query = """
SELECT purpose,
       COUNT(*) AS num_loans,
       AVG(CASE WHEN loan_status = 'Charged Off' THEN 1.0 ELSE 0 END) AS charge_off_rate
FROM loans
GROUP BY purpose
ORDER BY charge_off_rate DESC
"""
pd.read_sql(query, conn)

,purpose,num_loans,charge_off_rate
0,small_business,1828,0.259847
1,renewable_energy,103,0.184466
2,educational,325,0.172308
3,other,3993,0.158527
4,moving,583,0.157804
5,house,381,0.154856
6,medical,693,0.152958
7,debt_consolidation,18641,0.148436
8,vacation,381,0.139108
9,home_improvement,2976,0.116599


In [7]:
#car loans seem to have the highest interest rates
query = """
WITH ranked_dataset AS(
SELECT purpose, int_rate,
    RANK() OVER (PARTITION BY purpose ORDER BY int_rate DESC) AS ranking
FROM loans
ORDER BY purpose, ranking)

SELECT purpose, int_rate
FROM ranked_dataset
WHERE ranking = 1
"""

pd.read_sql(query, conn)

,purpose,int_rate
0,car,22.85
1,credit_card,24.11
2,credit_card,24.11
3,debt_consolidation,24.11
4,educational,21.27
5,home_improvement,23.91
6,house,23.13
7,major_purchase,23.59
8,medical,22.06
9,moving,22.11


In [8]:
#check, for each grade and employment length, what is the average loan amount
query = """
WITH by_grade_length_amnt AS (
    SELECT grade, emp_length, 
        AVG(loan_amnt) AS avg_loan_amnt
    FROM Loans
    GROUP BY grade, emp_length)
SELECT grade, emp_length, avg_loan_amnt,
    RANK() OVER (PARTITION BY grade ORDER BY avg_loan_amnt DESC) AS Ranked
FROM by_grade_length_amnt
"""

pd.read_sql(query, conn)

,grade,emp_length,avg_loan_amnt,Ranked
0,A,8 years,9477.216749,1
1,A,10+ years,9377.569686,2
2,A,9 years,9185.160819,3
3,A,5 years,8967.632850,4
4,A,7 years,8882.546083,5
...,...,...,...,...
79,G,3 years,19042.763158,8
80,G,7 years,17976.315789,9
81,G,2 years,17213.000000,10
82,G,1 year,16326.470588,11


In [ ]:
# calculate the average interest rate for each month and see how it changes over time.
# in general, there is an upward trend
# the initial 13.75% is suspicious, need to check how many loans were issued in that month
query = """
WITH monthly_avg AS (
    SELECT issue_d, AVG(int_rate) AS avg_rate
    FROM loans
    GROUP BY issue_d
)
SELECT issue_d, avg_rate,
    LAG(avg_rate) OVER (ORDER BY issue_d) AS prev_month_rate,
    avg_rate - LAG(avg_rate) OVER (ORDER BY issue_d) AS rate_change
FROM monthly_avg
ORDER BY issue_d
"""

pd.read_sql(query, conn)

,issue_d,avg_rate,prev_month_rate,rate_change
0,2007-06-01 00:00:00,13.750000,NaN,NaN
1,2007-07-01 00:00:00,9.254667,13.750000,-4.495333
2,2007-08-01 00:00:00,10.294848,9.254667,1.040182
3,2007-09-01 00:00:00,10.083889,10.294848,-0.210960
4,2007-10-01 00:00:00,10.806809,10.083889,0.722920
5,2007-11-01 00:00:00,9.798649,10.806809,-1.008160
6,2007-12-01 00:00:00,10.659176,9.798649,0.860528
7,2008-01-01 00:00:00,10.545380,10.659176,-0.113796
8,2008-02-01 00:00:00,10.742471,10.545380,0.197091
9,2008-03-01 00:00:00,11.143686,10.742471,0.401215


In [ ]:
# only one loan was issued in that month, which explains the suspiciously high interest rate.
query = """
SELECT issue_d, COUNT(*)
FROM loans
GROUP BY issue_d
"""

pd.read_sql(query, conn)

,issue_d,COUNT(*)
0,2007-06-01 00:00:00,1
1,2007-07-01 00:00:00,30
2,2007-08-01 00:00:00,33
3,2007-09-01 00:00:00,18
4,2007-10-01 00:00:00,47
5,2007-11-01 00:00:00,37
6,2007-12-01 00:00:00,85
7,2008-01-01 00:00:00,171
8,2008-02-01 00:00:00,174
9,2008-03-01 00:00:00,236


In [13]:
query = """
SELECT grade
FROM loans
WHERE issue_d LIKE '2007-06-01%'
"""
pd.read_sql(query, conn)

,grade
0,E
